# OAMNet — T4 GPU Training
**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `intensities.npy` and `spectra.npy` to Google Drive folder `oam_fog_data`
3. Run all cells (Runtime → Run all)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split

DATA_DIR = '/content/drive/MyDrive/oam_fog_data'
OUT_DIR  = '/content/drive/MyDrive/oam_fog_data/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class FogDataset(Dataset):
    def __init__(self, d):
        self.I = np.load(os.path.join(d, 'intensities.npy'))
        self.S = np.load(os.path.join(d, 'spectra.npy'))
        # Calculate sample-wise means for log-mean statistics
        sample_means = self.I.mean(axis=(1, 2))
        log_means = np.log10(sample_means + 1e-8)
        self.global_log_mean_mean = float(log_means.mean())
        self.global_log_mean_std = float(log_means.std())
        print(f'Loaded {len(self.I)} samples | mean={self.global_log_mean_mean:.4f} std={self.global_log_mean_std:.4f}')
    def __len__(self): return len(self.I)
    def __getitem__(self, i):
        intensity = self.I[i].astype(np.float32)
        mean = intensity.mean()
        std = intensity.std()
        intensity_norm = (intensity - mean) / (std + 1e-8)
        log_mean = np.log10(mean + 1e-8)
        aux = (log_mean - self.global_log_mean_mean) / (self.global_log_mean_std + 1e-8)
        return (
            torch.from_numpy(intensity_norm).unsqueeze(0),
            torch.tensor([aux], dtype=torch.float32),
            torch.from_numpy(self.S[i])
        )

def blk(a, b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,padding=1,bias=False), nn.BatchNorm2d(b), nn.ReLU(True),
        nn.Conv2d(b,b,3,padding=1,bias=False), nn.BatchNorm2d(b), nn.ReLU(True),
        nn.MaxPool2d(2))

class OAMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net  = nn.Sequential(blk(1,32), blk(32,64), blk(64,128), blk(128,256),
                                   nn.AdaptiveAvgPool2d(2), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(1024 + 1, 128), nn.ReLU(True),
                                   nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 11))
    def forward(self, x, aux):
        feat = self.net(x)
        feat_aux = torch.cat([feat, aux], dim=1)
        return torch.softmax(self.head(feat_aux), dim=1)

print('Classes defined.')

In [ ]:
W = torch.ones(11); W[6] = 0.1  # downweight dominant l=1 mode

def loss_fn(p, t):
    mse = (F.mse_loss(p, t, reduction='none') * W.to(p.device)).mean()
    pred_col = p[:, 6]
    target_col = t[:, 6]
    if pred_col.size(0) <= 1:
        pl = torch.tensor(0.0, device=p.device)
    else:
        pred_diff = pred_col - pred_col.mean()
        target_diff = target_col - target_col.mean()
        num = (pred_diff * target_diff).sum()
        denom = torch.sqrt((pred_diff ** 2).sum() * (target_diff ** 2).sum() + 1e-8)
        pl = 1.0 - (num / denom)
    return mse + 0.005 * pl

def pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a**2).sum() * (b**2).sum())
    return float(np.dot(a, b) / d) if d > 0 else 0.0

def train_ep(m, ld, opt):
    m.train(); tot = 0
    for x, aux, y in ld:
        x, aux, y = x.to(device), aux.to(device), y.to(device); opt.zero_grad()
        l = loss_fn(m(x, aux), y); l.backward(); opt.step(); tot += l.item()
    return tot / len(ld)

def val_ep(m, ld):
    m.eval(); tot = 0; rs = []
    with torch.no_grad():
        for x, aux, y in ld:
            p = m(x.to(device), aux.to(device))
            tot += loss_fn(p, y.to(device)).item()
            for i in range(len(p)): rs.append(pearson(p[i].cpu().numpy(), y[i].numpy()))
    return tot / len(ld), float(np.mean(rs))

print('Functions defined.')

In [ ]:
EPOCHS = 100; BS = 32
ds = FogDataset(DATA_DIR); n = len(ds)
nt, nv = int(.8*n), int(.1*n); nte = n - nt - nv
g = torch.Generator().manual_seed(42)
trset, vaset, teset = random_split(ds, [nt, nv, nte], generator=g)
trl = DataLoader(trset, BS, shuffle=True,  num_workers=2, pin_memory=True)
val = DataLoader(vaset, BS, shuffle=False, num_workers=2, pin_memory=True)
tel = DataLoader(teset, BS, shuffle=False, num_workers=2, pin_memory=True)

model = OAMNet().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
CKPT  = os.path.join(OUT_DIR, 'oamnet_best.pt')
best  = 1e9; log = []

for ep in range(1, EPOCHS+1):
    tr_l = train_ep(model, trl, opt)
    vl, _ = val_ep(model, val)
    sch.step()
    log.append({'epoch': ep, 'train_loss': tr_l, 'val_loss': vl})
    tag = ''
    if vl < best:
        best = vl; torch.save(model.state_dict(), CKPT); tag = ' <- best'
    print(f'Epoch {ep:03d} | train={tr_l:.6f} | val={vl:.6f}{tag}')

with open(os.path.join(OUT_DIR, 'losses.json'), 'w') as f: json.dump(log, f, indent=2)
model.load_state_dict(torch.load(CKPT))
tl, tr2 = val_ep(model, tel)
print(f'\nTest loss={tl:.6f} | Mean Pearson r={tr2:.4f}')

In [ ]:
# --- Evaluation figures ---
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()
P, T, Img = [], [], []
with torch.no_grad():
    for x, aux, y in tel:
        p = model(x.to(device), aux.to(device))
        P.append(p.cpu().numpy()); T.append(y.numpy()); Img.append(x.numpy())
P = np.concatenate(P); T = np.concatenate(T); Img = np.concatenate(Img)

# Fig 1: training curve
fig, ax = plt.subplots(figsize=(9,4))
ax.plot([d['epoch'] for d in log], [d['train_loss'] for d in log], label='Train')
ax.plot([d['epoch'] for d in log], [d['val_loss']   for d in log], label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted MSE'); ax.legend(); ax.grid(alpha=.3)
fig.savefig(os.path.join(OUT_DIR,'fig1_training_curve.png'), dpi=150); plt.show()

# Fig 2: example predictions
fig, axes = plt.subplots(4, 3, figsize=(13,16))
for i in range(4):
    axes[i,0].imshow(Img[i,0], cmap='inferno'); axes[i,0].axis('off')
    axes[i,0].set_title(f'Sample {i+1}: Intensity')
    axes[i,1].bar(range(11), T[i], color='steelblue')
    axes[i,1].set_title(f'Sample {i+1}: True'); axes[i,1].set_ylim(0,1)
    axes[i,2].bar(range(11), P[i], color='darkorange')
    axes[i,2].set_title(f'Sample {i+1}: Predicted'); axes[i,2].set_ylim(0,1)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR,'fig2_predictions.png'), dpi=150); plt.show()

# Fig 3: l=1 scatter
L = 6; tl, pl = T[:,L], P[:,L]; a, b = tl-tl.mean(), pl-pl.mean()
d = np.sqrt((a**2).sum()*(b**2).sum()); r = float(np.dot(a,b)/d) if d > 0 else 0.0
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(tl, pl, s=18, alpha=.5, label=f'r={r:.3f}')
lo, hi = min(tl.min(),pl.min())-.02, max(tl.max(),pl.max())+.02
ax.plot([lo,hi],[lo,hi],'r--'); ax.set_xlabel('True l=1'); ax.set_ylabel('Predicted l=1')
ax.legend(); ax.grid(alpha=.3); ax.set_aspect('equal')
fig.savefig(os.path.join(OUT_DIR,'fig3_l1_scatter.png'), dpi=150); plt.show()
print(f'l=1 Pearson r = {r:.4f}')
print(f'All outputs saved to: {OUT_DIR}')